# **Masked Self-Attention in Transformer Decoder**
### *The Engine of Modern Auto-Regressive Language Models (GPT, LLaMA, Claude)*

Go to the link below for detailed information on Masked Self-Attention:<br>
**LINK :** https://youtu.be/m6onaKFzF94?si=GUZUDfqA-ZOaSNPB

---

## 1. What is Masked Self-Attention?

**Masked Self-Attention** (also known as **Causal Attention**) is a specialized version of the standard self-attention mechanism used exclusively inside the **Transformer Decoder**.

* In the **Encoder**, words can look in **both directions** (past and future words) to understand full context.
* In the **Decoder**, words are **strictly forbidden from looking ahead into future words**. Each word is only permitted to attend to itself and previously generated words.

## 2. Why is Masked Self-Attention Needed? (The "Cheating" Problem)

To understand why we need masking, we have to look at the difference between **Training** and **Inference (Testing)**.

### **A. At Inference Time (Generating Text in the Real World)**
When a model like ChatGPT generates text, it works **auto-regressively** (one word at a time):
1. It receives: `"The cat sat on the"`
2. It predicts the next word: `"mat"`
3. It appends `"mat"` and predicts the next word.

At step 2, the word `"mat"` **does not exist yet in the physical universe**. The model cannot look into the future because the future hasn't happened!

---

### **B. During Training (The Parallel Training Paradox)**
In deep learning, training one word at a time sequentially would be as painfully slow as an old RNN/LSTM.

To train at blazing GPU speeds, we use **Teacher Forcing**: we feed the entire target sentence into the decoder **all at once in parallel**:
$$\text{Target Input: } [\text{"<start>"}, \; \text{"I"}, \; \text{"love"}, \; \text{"deep"}, \; \text{"learning"}]$$

### **The Fatal Flaw without Masking:**
If we used standard self-attention here, when the decoder is trying to learn how to predict `"love"` (given `"I"`), it could simply **look ahead to position 3 and copy the answer `"love"`**!

* The model would **cheat** by looking at future tokens.
* During training, its loss would drop to almost zero.
* But during real-world inference (where future words don't exist), the model would completely fail because it never learned how to predict without cheating.

> **The Solution:** Masking artificially blinds the decoder to all future positions during parallel training, ensuring that position $t$ can only attend to positions $\le t$.

## 3. How Masking Works: The $-\infty$ (Negative Infinity) Softmax Trick

Masked attention implements this restriction using a mathematical property of the **Softmax** function.

### **The Core Math Formula:**

$$\text{MaskedAttention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right) V$$

Where $M$ is the **Look-Ahead Mask Matrix**.

---

### **The Mathematical Trick:**
Recall the Softmax formula for an element $z_i$:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum e^{z_j}}$$

What happens when an input to softmax is **negative infinity ($-\infty$)**?

$$e^{-\infty} = \frac{1}{e^{\infty}} = \mathbf{0}$$

By adding **$-\infty$** to all future positions in the attention score matrix before passing through softmax, **their attention probability is forced to exactly 0.0%**!

```text
Raw Dot-Product Scores            Look-Ahead Mask (M)             After Softmax
      w1   w2   w3                     w1   w2   w3                 w1    w2    w3
w1 [ 4.2  1.5  0.8 ]             w1 [  0   -inf -inf ]        w1 [ 1.0   0.0   0.0 ]
w2 [ 2.1  5.0  1.2 ]     +       w2 [  0    0   -inf ]   ──►  w2 [ 0.15  0.85  0.0 ]
w3 [ 1.1  3.4  4.8 ]             w3 [  0    0    0   ]        w3 [ 0.05  0.25  0.70]
```

* **Row 1 (`w1`):** Can only attend to `w1` (100%). Future words `w2, w3` are forced to **0%**.
* **Row 2 (`w2`):** Can attend to `w1` (15%) and `w2` (85%). Future word `w3` is forced to **0%**.
* **Row 3 (`w3`):** Can attend to all previous words `w1, w2, w3`.

## 4. Step-by-Step Mathematical Walkthrough

Let's walk through the math with a 3-word target sentence:
$$[\text{"<start>"}, \; \text{"I"}, \; \text{"love"}]$$

### **Step 1: Compute Raw Attention Scores ($\frac{QK^\top}{\sqrt{d_k}}$)**
Suppose our scaled dot-product produces the following $3 \times 3$ similarity matrix:

$$S = \begin{bmatrix}
12 & 8 & 4 \\
6 & 10 & 5 \\
3 & 7 & 11
\end{bmatrix}$$

### **Step 2: Construct the Upper-Triangular Mask Matrix ($M$)**
We create a mask where all illegal future positions (above the main diagonal) are filled with $-\infty$, and valid past positions are $0$:

$$M = \begin{bmatrix}
0 & -\infty & -\infty \\
0 & 0 & -\infty \\
0 & 0 & 0
\end{bmatrix}$$

### **Step 3: Add Mask to Scores ($S + M$)**

$$S_{\text{masked}} = S + M = \begin{bmatrix}
12 & -\infty & -\infty \\
6 & 10 & -\infty \\
3 & 7 & 11
\end{bmatrix}$$

### **Step 4: Apply Softmax Row-by-Row**

* **Row 1:** $\text{softmax}([12, \; -\infty, \; -\infty]) = [\frac{e^{12}}{e^{12} + 0 + 0}, \; \frac{0}{e^{12}}, \; \frac{0}{e^{12}}] = \mathbf{[1.0, \; 0.0, \; 0.0]}$
* **Row 2:** $\text{softmax}([6, \; 10, \; -\infty]) = [\frac{e^6}{e^6 + e^{10} + 0}, \; \frac{e^{10}}{e^6 + e^{10} + 0}, \; 0.0] \approx \mathbf{[0.018, \; 0.982, \; 0.0]}$
* **Row 3:** $\text{softmax}([3, \; 7, \; 11]) \approx \mathbf{[0.0003, \; 0.018, \; 0.9817]}$

$$\text{Attention Weights } A = \begin{bmatrix}
1.000 & 0.000 & 0.000 \\
0.018 & 0.982 & 0.000 \\
0.000 & 0.018 & 0.982
\end{bmatrix}$$

### **Step 5: Multiply by Values ($A \times V$)**
Now we multiply these probabilities by the Value vectors $V$. The resulting vector for each word contains **zero influence** from any future token.

## 5. Comparison: The 3 Types of Attention in Transformers

A full Transformer architecture uses attention in three distinct places. Here is how they compare:

| Feature | 1. Self-Attention (Encoder) | 2. Masked Self-Attention (Decoder) | 3. Cross-Attention (Decoder) |
| :--- | :--- | :--- | :--- |
| **Where it Lives** | Encoder Stack | Decoder Stack (Layer 1) | Decoder Stack (Layer 2) |
| **Where Q, K, V Come From** | All from **Encoder Input** ($Q=K=V$) | All from **Decoder Targets** ($Q=K=V$) | $Q$ from **Decoder**, $K, V$ from **Encoder** |
| **Masking Applied?** | ❌ **No** | ✅ **Yes** ($-\infty$ future mask) | ❌ **No** (Decoder can see all source words) |
| **Directionality** | **Bidirectional** (looks left & right) | **Causal / Unidirectional** (looks only left) | **Cross-Modal** (aligns target with full source) |
| **Purpose** | Understand full sentence context | Predict next token without cheating | Extract relevant translation from source sentence |

## 6. Python Implementations: From Scratch & Frameworks

### **A. TensorFlow & Keras Implementation (Recommended)**

In Keras, you can apply Masked Self-Attention effortlessly using `layers.MultiHeadAttention` with the parameter **`use_causal_mask=True`**:

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Define MultiHeadAttention layer with causal/masked attention enabled
# num_heads=2, key_dim=4
causal_mha = layers.MultiHeadAttention(num_heads=2, key_dim=4)

# 2. Simulated input: (batch_size=1, sequence_length=3, embedding_dim=4)
# Represents 3 tokens: ["<start>", "I", "love"]
inputs = tf.random.normal((1, 3, 4))

# 3. Compute Masked Self-Attention
# Passing use_causal_mask=True automatically applies the -inf upper-triangular mask!
output, attention_weights = causal_mha(
    query=inputs,
    value=inputs,
    key=inputs,
    use_causal_mask=True,
    return_attention_scores=True
)

print("Output shape (Batch, Seq_Len, Dim):", output.shape)
print("Attention Weights shape (Batch, Heads, Target_Len, Source_Len):", attention_weights.shape)

# Notice that the upper-triangular values are strictly 0.0!
print("\nMasked Attention Matrix for Head 0:")
print(tf.round(attention_weights[0, 0], 4).numpy())
```

---

### **B. Building a Complete Keras Decoder Block**

Here is how Masked Self-Attention is plugged into a full Decoder block in Keras:

```python
class TransformerDecoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        # 1. Masked Self-Attention (Causal)
        self.causal_mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        # 2. Feed-Forward Network
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        # 3. Layer Normalization & Dropout
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, x, training=False):
        # Step 1: Masked Self-Attention with Causal Mask
        attn_output = self.causal_mha(query=x, value=x, key=x, use_causal_mask=True)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)  # Add & Norm
        
        # Step 2: Feed Forward
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)  # Add & Norm

# Test Decoder Block
decoder_block = TransformerDecoderBlock(embed_dim=16, num_heads=2, ff_dim=64)
sample_seq = tf.random.normal((1, 5, 16))
print("Decoder Block Output Shape:", decoder_block(sample_seq).shape)
```

---

### **C. NumPy Implementation From Scratch**

```python
import numpy as np

def masked_self_attention(Q, K, V):
    seq_len, d_k = Q.shape
    
    # 1. Compute Scaled Dot-Product Scores
    scores = np.dot(Q, K.T) / np.sqrt(d_k)
    
    # 2. Create Upper-Triangular Mask (True above diagonal)
    mask = np.triu(np.ones((seq_len, seq_len)), k=1).astype(bool)
    
    # 3. Fill future positions with -inf
    scores[mask] = -np.inf
    
    # 4. Softmax row-wise
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # 5. Multiply by Values
    output = np.dot(attention_weights, V)
    return output, attention_weights

# Test with 3 tokens (dimension = 4)
np.random.seed(42)
Q = K = V = np.random.randn(3, 4)
output, weights = masked_self_attention(Q, K, V)
print("NumPy Masked Attention Weights:\n", np.round(weights, 4))
```

---

### **D. PyTorch Implementation**

```python
import torch
import torch.nn.functional as F

# Batch size = 1, Sequence length = 4, Dimension = 8
q = k = v = torch.randn(1, 4, 8)

# PyTorch built-in causal masking
output = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("PyTorch Causal Attention Output shape:", output.shape)
```

---

### **Summary of Masked Self-Attention**
1. **What it does:** Allows the decoder to calculate attention without attending to future words.
2. **Why it is needed:** Enables fast parallel GPU training with Teacher Forcing while preventing the model from "cheating" by reading future words.
3. **How it works:** Adds $-\infty$ above the diagonal of the $QK^\top$ score matrix, forcing future softmax probabilities to exactly $0$.